# Baseline Models for Anomaly Detection

### TODO
- evaluate baselines on test data

## Import Data

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# data is that of the training and scaled dataset
df = pd.read_csv('train_scaled.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)


### Set Parameters

In [8]:
brands = ['Apple', 'Amazon', 'Salesforce', 'CVS', 'Facebook', 'Google', 'IBM', 'Coca-Cola', 'Pfizer', 'UPS']

WINDOW = 288 # 24 hours of 5-minute rows
THRESHOLD = 3 # Z-score threshold for anomaly detection
STD_FLOOR = 0.1 # Minimum standard deviation to avoid division by zero

## Z-Score

### Compute Rolling Z-Score Per Brand

In [9]:
results = []

for brand in brands:
    grp = df[['timestamp', brand]].copy()
    grp.columns = ['timestamp', 'value']
    grp['brand'] = brand

    roll_mean = grp['value'].rolling(window=WINDOW, min_periods=WINDOW // 2).mean()
    roll_std = grp['value'].rolling(window=WINDOW, min_periods=WINDOW // 2).std().clip(lower=STD_FLOOR)

    grp['rolling_mean'] = roll_mean
    grp['rolling_std'] = roll_std
    grp['z_score'] = (grp['value'] - roll_mean) / roll_std

    grp['is_anomaly'] = grp['z_score'].abs() > THRESHOLD
    
    results.append(grp)

df_all = pd.concat(results, ignore_index=True)

In [10]:
for brand in brands:
    sub = df_all[df_all['brand'] == brand]
    n = sub['is_anomaly'].sum()
    rate = n / len(sub) * 100
    maxz = sub['z_score'].abs().max()
    print(f"{brand:<15} {n} anomalies ({rate:.1f}%) max z-score: {maxz:.1f}")

Apple           175 anomalies (1.6%) max z-score: 16.9
Amazon          111 anomalies (1.0%) max z-score: 16.3
Salesforce      224 anomalies (2.0%) max z-score: 15.5
CVS             249 anomalies (2.2%) max z-score: 16.6
Facebook        172 anomalies (1.5%) max z-score: 16.6
Google          251 anomalies (2.3%) max z-score: 13.1
IBM             191 anomalies (1.7%) max z-score: 14.1
Coca-Cola       205 anomalies (1.8%) max z-score: 16.3
Pfizer          243 anomalies (2.2%) max z-score: 12.5
UPS             279 anomalies (2.5%) max z-score: 16.8


## Compute Moving Average & Residual

In [4]:
results = []

for brand in brands:
    grp = df[['timestamp', brand]].copy()
    grp.columns = ['timestamp', 'value']
    grp['brand'] = brand

    grp['moving_avg'] = grp['value'].rolling(window=WINDOW, min_periods=WINDOW // 2).mean()

    grp['residual'] = grp['value'] - grp['moving_avg']

    residual_std = grp['residual'].rolling(window=WINDOW, min_periods=WINDOW // 2).std().clip(lower=STD_FLOOR)

    grp['residual_std'] = residual_std

    grp['is_anomaly'] = grp['residual'].abs() > (THRESHOLD * residual_std)

    results.append(grp)

df_all = pd.concat(results, ignore_index=True)

In [5]:
for brand in brands:
    sub = df_all[df_all['brand'] == brand]
    n = sub['is_anomaly'].sum()
    rate = n / len(sub) * 100
    max_residual = sub['residual'].abs().max()
    print(f"{brand:<15} {n} anomalies ({rate:.1f}%) max residual: {max_residual:.1f}")

Apple           170 anomalies (1.5%) max residual: 46.7
Amazon          111 anomalies (1.0%) max residual: 50.0
Salesforce      237 anomalies (2.1%) max residual: 40.2
CVS             249 anomalies (2.2%) max residual: 48.4
Facebook        165 anomalies (1.5%) max residual: 58.5
Google          250 anomalies (2.2%) max residual: 21.5
IBM             197 anomalies (1.8%) max residual: 18.2
Coca-Cola       205 anomalies (1.8%) max residual: 30.7
Pfizer          248 anomalies (2.2%) max residual: 11.8
UPS             273 anomalies (2.5%) max residual: 9.6


## ARIMA

### Check If Stationary (no long term trend)

In [4]:
from statsmodels.tsa.stattools import adfuller

for brand in brands:
    series = df[brand].dropna()
    result = adfuller(series)
    p_value = result[1]
    status = "stationary" if p_value < 0.05 else "non-stationary"
    print(f"{brand:<15} ADF p-value: {p_value:.4f} ({status})")

Apple           ADF p-value: 0.0000 (stationary)
Amazon          ADF p-value: 0.0000 (stationary)
Salesforce      ADF p-value: 0.0000 (stationary)
CVS             ADF p-value: 0.0000 (stationary)
Facebook        ADF p-value: 0.0000 (stationary)
Google          ADF p-value: 0.0000 (stationary)
IBM             ADF p-value: 0.0000 (stationary)
Coca-Cola       ADF p-value: 0.0000 (stationary)
Pfizer          ADF p-value: 0.0000 (stationary)
UPS             ADF p-value: 0.0000 (stationary)


### Fit ARIMA

In [5]:
from statsmodels.tsa.arima.model import ARIMA

results = []

for brand in brands:
    print(f"Fitting ARIMA for {brand}...")

    series = df[brand].copy()
    split_idx = int(len(series) * 0.65)
    train = series.iloc[:split_idx]

    model = ARIMA(train, order=(2, 1, 2))
    fitted_model = model.fit()

    train_fitted = fitted_model.fittedvalues

    predict_full = fitted_model.predict(
        start=0, 
        end=len(series)-1, dynamic=False
    )

    grp = df[['timestamp', brand]].copy()
    grp.columns = ['timestamp', 'value']
    grp['brand'] = brand
    grp['predicted'] = predict_full.values
    grp['residual'] = grp['value'] - grp['predicted']

    train_residuals = grp['residual'].iloc[:split_idx]
    residual_std = train_residuals.std()

    grp['residual_std'] = residual_std
    grp['is_anomaly'] = grp['residual'].abs() > (THRESHOLD * residual_std)
    grp['window'] = ['fit' if i < split_idx else 'detect' for i in range(len(grp))]

    results.append(grp)
    print(f"{grp['is_anomaly'].sum()} anomalies detected")

df_all = pd.concat(results, ignore_index=True)


Fitting ARIMA for Apple...
123 anomalies detected
Fitting ARIMA for Amazon...
50 anomalies detected
Fitting ARIMA for Salesforce...


/opt/miniconda3/envs/tw_ts_env/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


241 anomalies detected
Fitting ARIMA for CVS...
134 anomalies detected
Fitting ARIMA for Facebook...
141 anomalies detected
Fitting ARIMA for Google...


/opt/miniconda3/envs/tw_ts_env/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


272 anomalies detected
Fitting ARIMA for IBM...
207 anomalies detected
Fitting ARIMA for Coca-Cola...
151 anomalies detected
Fitting ARIMA for Pfizer...
191 anomalies detected
Fitting ARIMA for UPS...
151 anomalies detected


/opt/miniconda3/envs/tw_ts_env/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/miniconda3/envs/tw_ts_env/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [6]:
print("=" * 65)
print(f"{'Brand':<15} {'Anomalies':>10} {'Rate':>8} {'Residual Std':>14}")
print("=" * 65)
for brand in brands:
    sub  = df_all[df_all['brand'] == brand]
    n    = sub['is_anomaly'].sum()
    rate = n / len(sub) * 100
    std  = sub['residual_std'].iloc[0]
    print(f"{brand:<15} {n:>10} {rate:>7.1f}%  {std:>14.4f}")
print("=" * 65)

Brand            Anomalies     Rate   Residual Std
Apple                  123     1.1%          0.4306
Amazon                  50     0.4%          0.8772
Salesforce             241     2.2%          0.7032
CVS                    134     1.2%          0.9258
Facebook               141     1.3%          0.6163
Google                 272     2.4%          0.6548
IBM                    207     1.9%          0.7399
Coca-Cola              151     1.4%          0.7516
Pfizer                 191     1.7%          0.9158
UPS                    151     1.4%          0.4949


## STL Decomposition

### Fit STL

In [9]:
from statsmodels.tsa.seasonal import STL

results = []

for brand in brands:
    print(f"Fitting STL for {brand}...")

    grp = df[['timestamp', brand]].copy()
    grp.columns = ['timestamp', 'value']
    grp['brand'] = brand

    split_idx = int(len(grp) * 0.65)
    train = grp['value'].iloc[:split_idx]

    stl = STL(train, period=WINDOW, robust=True)
    stl_fit = stl.fit()

    trend_train = stl_fit.trend
    seasonal_train = stl_fit.seasonal
    residual_train = stl_fit.resid

    # expected = trend + seasonal pattern learned from training
    # we repeat the seasonal pattern to cover the full series length
    season_period = seasonal_train.values[-WINDOW:]
    n_full = len(grp)
    n_repeats = int(np.ceil(n_full / WINDOW)) + 1
    seasonal_full = np.tile(season_period, n_repeats)[:n_full]

    # extend trend using last known slope
    last_trend_val = trend_train.values[-1]
    last_trend_diff = np.diff(trend_train.values[-10:]).mean()
    n_detect = n_full - split_idx
    trend_extension = last_trend_val + last_trend_diff * np.arange(1, n_detect + 1)
    trend_full = np.concatenate([trend_train.values, trend_extension])

    grp['trend'] = trend_full
    grp['seasonal'] = seasonal_full
    grp['expected'] = grp['trend'] + grp['seasonal']
    grp['residual'] = grp['value'] - grp['expected']

    residual_std = residual_train.std()
    grp['residual_std'] = residual_std
    grp['is_anomaly'] = grp['residual'].abs() > (THRESHOLD * residual_std)
    grp['window'] = ['fit' if i < split_idx else 'detect' for i in range(len(grp))]

    results.append(grp)
    print(f"{grp['is_anomaly'].sum()} anomalies detected")

df_all = pd.concat(results, ignore_index=True)

Fitting STL for Apple...
96 anomalies detected
Fitting STL for Amazon...
2901 anomalies detected
Fitting STL for Salesforce...
3303 anomalies detected
Fitting STL for CVS...
243 anomalies detected
Fitting STL for Facebook...
3554 anomalies detected
Fitting STL for Google...
2923 anomalies detected
Fitting STL for IBM...
3010 anomalies detected
Fitting STL for Coca-Cola...
2496 anomalies detected
Fitting STL for Pfizer...
3176 anomalies detected
Fitting STL for UPS...
255 anomalies detected


In [10]:
print("=" * 65)
print(f"{'Brand':<15} {'Anomalies':>10} {'Rate':>8} {'Residual Std':>14}")
print("=" * 65)
for brand in brands:
    sub  = df_all[df_all['brand'] == brand]
    n    = sub['is_anomaly'].sum()
    rate = n / len(sub) * 100
    std  = sub['residual_std'].iloc[0]
    print(f"{brand:<15} {n:>10} {rate:>7.1f}%  {std:>14.4f}")
print("=" * 65)

Brand            Anomalies     Rate   Residual Std
Apple                   96     0.9%          0.6415
Amazon                2901    26.1%          0.8143
Salesforce            3303    29.7%          0.4908
CVS                    243     2.2%          0.9313
Facebook              3554    31.9%          0.6115
Google                2923    26.3%          0.8013
IBM                   3010    27.0%          0.7428
Coca-Cola             2496    22.4%          0.7784
Pfizer                3176    28.5%          0.9398
UPS                    255     2.3%          0.8536


## Isolation Forest

In [13]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

CONTAMINATION = 0.02

In [14]:
def build_features(grp, brand):
    feat = pd.DataFrame()

    # ── 1. Raw scaled value ───────────────────────────────────────
    feat['value'] = grp[brand]

    # ── 2. Rolling statistics (local context) ────────────────────
    feat['rolling_mean_1d'] = grp[brand].rolling(288,  min_periods=144).mean()
    feat['rolling_std_1d']  = grp[brand].rolling(288,  min_periods=144).std()
    feat['rolling_mean_6h'] = grp[brand].rolling(72,   min_periods=36).mean()
    feat['rolling_std_6h']  = grp[brand].rolling(72,   min_periods=36).std()

    # ── 3. Residual from rolling mean (like MA residual) ─────────
    feat['residual_1d'] = feat['value'] - feat['rolling_mean_1d']

    # ── 4. Rate of change ─────────────────────────────────────────
    feat['diff_1']  = grp[brand].diff(1)    # change from last 5 min
    feat['diff_12'] = grp[brand].diff(12)   # change from last hour

    # ── 5. Time-based features (encode seasonality manually) ─────
    feat['hour']        = grp['timestamp'].dt.hour
    feat['day_of_week'] = grp['timestamp'].dt.dayofweek   # 0=Mon, 6=Sun
    feat['is_weekend']  = (feat['day_of_week'] >= 5).astype(int)

    # ── 6. Cyclical encoding of hour (prevents 23→0 discontinuity)
    feat['hour_sin'] = np.sin(2 * np.pi * feat['hour'] / 24)
    feat['hour_cos'] = np.cos(2 * np.pi * feat['hour'] / 24)

    return feat.dropna()

In [15]:
results = []

for brand in brands:
    print(f"Fitting Isolation Forest for {brand}...")

    grp = df[['timestamp', brand]].copy()

    features = build_features(grp, brand)
    valid_idx = features.index

    split_idx = int(len(features) * 0.65)

    X_train = features.iloc[:split_idx]
    X_full  = features

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_full_scaled  = scaler.transform(X_full)

    model = IsolationForest(
        n_estimators=200,
        contamination=CONTAMINATION,
        random_state=42,
        max_samples='auto',
    )
    model.fit(X_train_scaled)

    preds = model.predict(X_full_scaled)
    scores = model.score_samples(X_full_scaled)

    result = grp.loc[valid_idx].copy()
    result.columns = ['timestamp', 'value']
    result['brand'] = brand
    result['anomaly_score'] = scores
    result['is_anomaly'] = preds == -1
    result['window'] = ['fit' if i < split_idx else 'detect' for i in range(len(result))]

    results.append(result)
    n = result['is_anomaly'].sum()
    print(f"{n} anomalies detected ({n / len(result) * 100:.1f}%)")

df_all = pd.concat(results, ignore_index=True)

Fitting Isolation Forest for Apple...
317 anomalies detected (2.9%)
Fitting Isolation Forest for Amazon...
171 anomalies detected (1.6%)
Fitting Isolation Forest for Salesforce...
402 anomalies detected (3.7%)
Fitting Isolation Forest for CVS...
325 anomalies detected (3.0%)
Fitting Isolation Forest for Facebook...
182 anomalies detected (1.7%)
Fitting Isolation Forest for Google...
338 anomalies detected (3.1%)
Fitting Isolation Forest for IBM...
347 anomalies detected (3.2%)
Fitting Isolation Forest for Coca-Cola...
214 anomalies detected (1.9%)
Fitting Isolation Forest for Pfizer...
167 anomalies detected (1.5%)
Fitting Isolation Forest for UPS...
177 anomalies detected (1.6%)


In [16]:
print("=" * 65)
print(f"{'Brand':<15} {'Anomalies':>10} {'Rate':>8} {'Min Score':>12}")
print("=" * 65)
for brand in brands:
    sub  = df_all[df_all['brand'] == brand]
    n    = sub['is_anomaly'].sum()
    rate = n / len(sub) * 100
    min_score = sub['anomaly_score'].min()
    print(f"{brand:<15} {n:>10} {rate:>7.1f}%  {min_score:>12.4f}")
print("=" * 65)

Brand            Anomalies     Rate    Min Score
Apple                  317     2.9%       -0.7853
Amazon                 171     1.6%       -0.7375
Salesforce             402     3.7%       -0.7655
CVS                    325     3.0%       -0.7517
Facebook               182     1.7%       -0.7335
Google                 338     3.1%       -0.7526
IBM                    347     3.2%       -0.7148
Coca-Cola              214     1.9%       -0.7537
Pfizer                 167     1.5%       -0.6999
UPS                    177     1.6%       -0.7076
